# Multi-Store Stock-Allocation Optimisation

A real mathematical-programming layer on top of the demand model. The forecast and the
classic EOQ / safety-stock analytics tell us what each SKU *ideally* wants to stock; this
notebook decides what each store-product actually **gets** when the procurement budget and
the warehouse are finite — by solving a **linear program (LP)** with PuLP + CBC.

The runnable model lives in [`optimize_allocation.py`](../optimize_allocation.py); here we
explain the formulation and reproduce the result.


In [1]:
import os
# Run from the repository root so results/ paths resolve.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import json
import numpy as np
import pandas as pd
import pulp
from optimize_allocation import load_skus, optimize_allocation, ABC_WEIGHT
print('PuLP', pulp.__version__, '| CBC available:', pulp.PULP_CBC_CMD(msg=0).available())

PuLP 3.3.2 | CBC available: /Users/mac/Demand And Stock Analysis ML  /.venv/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc


## 1. Decision variable

For each store-product $i$ (100 SKUs across 5 stores) we choose an **allocation**

$$x_i \ge 0 \quad\text{= units of stock to place at store-product } i.$$


## 2. Per-SKU parameters

Over a planning horizon of $H$ days:

| Symbol | Meaning | Formula |
|---|---|---|
| $\mu_i$ | expected demand over the horizon | `avg_daily_demand` $\times\,H$ |
| $\sigma_i$ | demand std over the horizon | `demand_std` $\times\sqrt{H}$ |
| $\tau_i$ | **service target** (order-up-to) | $\mu_i + z\,\sigma_i$ |
| $p_i$ | unit price | `unit_price` |
| $h_i$ | holding cost / unit / horizon | `holding_rate` $\times p_i \times H/365$ |
| $w_i$ | ABC importance weight | A=1.0, B=0.6, C=0.35 |
| $b_i$ | stockout penalty / unit short | `penalty_mult` $\times p_i \times w_i$ |

The safety term $z\,\sigma_i$ is what turns a mean forecast into a **service-level** target:
$z=1.64$ targets a ~95% cycle service level.


In [2]:
df = load_skus(horizon=7, service_z=1.64)
df[['Store ID','Product ID','abc_class','mu','sigma','tau','price','w']].head()

,Store ID,Product ID,abc_class,mu,sigma,tau,price,w
0,S003,P0011,A,744.66,112.788378,929.632941,119.772474,1.0
1,S003,P0008,A,761.95,101.808510,928.915957,112.518803,1.0
2,S004,P0009,A,756.07,105.486105,929.067212,112.048237,1.0
3,S005,P0009,A,848.96,125.673187,1055.064027,95.849039,1.0
4,S003,P0015,A,831.95,133.636899,1051.114514,97.691605,1.0


## 3. Objective — minimise total expected cost

$$\min_{x}\; \sum_i \Big(\underbrace{h_i\,x_i}_{\text{holding}} \;+\; \underbrace{b_i\,s_i}_{\text{stockout penalty}}\Big)$$

where $s_i \ge \tau_i - x_i,\; s_i \ge 0$ is the **shortfall** below the service target.
Holding cost pulls every $x_i$ **down**; the stockout penalty pushes it **up** toward
$\tau_i$. Because $b_i \gg h_i$, without limits every SKU would go straight to its target —
the constraints are what make allocation a real decision.


## 4. Constraints

$$\begin{aligned}
s_i &\ge \tau_i - x_i, \quad s_i \ge 0 && \text{(shortfall definition)}\\
\underbrace{m\,\tau_i}_{\text{min service}} \le\; & x_i \;\le\; \underbrace{\tau_i}_{\text{no overstock}} && \text{(per-SKU service floor)}\\
\sum_i p_i\,x_i &\le B && \text{(procurement budget)}\\
\sum_i x_i &\le C && \text{(warehouse capacity)}
\end{aligned}$$

* **Minimum service level** $m$: every SKU must receive at least a fraction $m$ of its
  target $\tau_i$ (default $m=0.5$).
* **Budget** $B$ and **capacity** $C$ are the shared, binding resources. When $B$ is
  tight the LP keeps high-value / high-weight A-class SKUs at full service and sacrifices
  cheap C-class coverage first — provably the cheapest way to absorb the shortfall.


In [3]:
# Size the budget/capacity so they actually bind (80% of the full-target cost).
full_cost  = float((df['price'] * df['tau']).sum())
full_units = float(df['tau'].sum())
budget   = 0.80 * full_cost
capacity = 0.90 * full_units

alloc, summary = optimize_allocation(df, budget=budget, capacity=capacity,
                                     min_service=0.5, holding_rate=0.25, penalty_mult=1.5)
print('Solver status :', summary['solver_status'])
print(f"Budget used   : {summary['budget_used']:,.0f} / {summary['budget']:,.0f}")
print(f"Capacity used : {summary['capacity_used']:,} / {summary['capacity']:,} units")
print(f"Total cost    : {summary['total_cost']:,.0f}  "
      f"(holding {summary['holding_cost']:,.0f} + penalty {summary['stockout_penalty']:,.0f})")
print(f"Avg service   : {summary['avg_service_fill_pct']}%  "
      f"({summary['skus_at_full_target']}/{summary['n_skus']} SKUs at full target)")
print(f"Savings vs even-cut baseline: {summary['savings_vs_baseline']:,.0f} "
      f"({summary['savings_pct']}%)")

Solver status : Optimal
Budget used   : 4,702,926 / 4,702,926
Capacity used : 67,162 / 82,351 units
Total cost    : 1,370,182  (holding 22,548 + penalty 1,347,634)
Avg service   : 72.2%  (44/100 SKUs at full target)
Savings vs even-cut baseline: 249,578 (15.4%)


## 5. The allocation plan

The biggest recommended replenishment orders (`recommended_order = allocation − current`).
Notice A-class SKUs sit at 100% fill while the budget shortfall is pushed onto lower classes.


In [4]:
alloc.sort_values('recommended_order', ascending=False)[
    ['Store ID','Product ID','abc_class','service_target','allocation',
     'current_inventory','recommended_order','service_fill_pct']].head(10)

,Store ID,Product ID,abc_class,service_target,allocation,current_inventory,recommended_order,service_fill_pct
0,S002,P0011,A,1040.5,1040,73,967,100.0
1,S003,P0019,A,1058.7,1059,108,951,100.0
2,S001,P0018,A,1067.1,1067,130,937,100.0
3,S002,P0010,A,1070.6,1071,143,928,100.0
4,S001,P0002,A,1060.8,1061,136,925,100.0
5,S005,P0008,A,1075.9,1076,154,922,100.0
6,S005,P0010,A,1036.2,1036,115,921,100.0
7,S005,P0009,A,1055.1,1055,136,919,100.0
8,S001,P0009,A,1042.5,1042,128,914,100.0
9,S002,P0009,A,1060.9,1061,153,908,100.0


In [5]:
# Service fill by ABC class — the optimiser protects the valuable SKUs first.
alloc.groupby('abc_class')['service_fill_pct'].mean().round(1)

abc_class
A    87.6
B    50.0
C    50.0
Name: service_fill_pct, dtype: float64

## 6. Why optimise? (vs an even cut)

A naive planner facing the same budget might shrink *every* target by the same factor.
That even cut is class-blind: it under-serves an A-class SKU exactly as much as a C-class
one. The LP instead concentrates the shortfall where it is cheapest, which is why its total
expected cost is lower:


In [6]:
print(f"Even-cut baseline cost : {summary['baseline_cost']:,.0f}")
print(f"Optimised cost         : {summary['total_cost']:,.0f}")
print(f"Saving                 : {summary['savings_vs_baseline']:,.0f} ({summary['savings_pct']}%)")

Even-cut baseline cost : 1,619,760
Optimised cost         : 1,370,182
Saving                 : 249,578 (15.4%)


## 7. Sensitivity — tightening the budget

Re-solving across budget levels traces the classic **cost vs budget** trade-off curve:
less budget → more shortfall penalty → higher total cost and lower service.


In [7]:
rows = []
for frac in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    _, s = optimize_allocation(df, budget=frac*full_cost, capacity=capacity, min_service=0.5)
    rows.append({'budget_frac': frac, 'service_%': s['avg_service_fill_pct'],
                 'total_cost': s['total_cost'], 'stockout_penalty': s['stockout_penalty']})
pd.DataFrame(rows)

,budget_frac,service_%,total_cost,stockout_penalty
0,0.5,50.0,4007122.08,3993029.41
1,0.6,56.8,3128142.09,3111230.89
2,0.7,65.2,2249162.09,2229432.35
3,0.8,72.2,1370182.08,1347633.81
4,0.9,79.9,497191.08,471824.28
5,1.0,90.6,128759.25,101419.20


---
**Outputs written by `optimize_allocation.py`** (consumed by the dashboard, chatbot tool
`optimizasyon_onerisi`, and the `/api/optimization` endpoint):

* `results/optimization_allocation.csv` — per-SKU allocation + costs
* `results/optimization_summary.json` — objective, budget/capacity use, service, savings
* `results/18_optimization_allocation.png` — allocation vs current chart
